In [6]:
import numpy as np
import os

os.chdir(r"C:\Users\Mohan2\Downloads\casp12.tar\casp12")

a = np.load('HMM_params.npz')
TM = np.log(a['np_TM']) #transition matrix values (log values), shape 3x3
EM = np.log(a['np_EM']) #  emission matrix values (log values), shape 21x3
null_dist = np.log(np.array([0.05,0.05,0.99])).reshape(3,1)

# AA_dict = {i:j for i,j in zip(['A','R','N','D','C','Q','E','G','H','I','L','K','M','F','P','S','T','W','Y','V','X'],range(21))}
# SS_dict = {'H': 0, 'E': 1, 'L':2}

# def read_fasta(file):
#     with open(file, "r") as o:
#         for line in o:
#             if line[0]==">":
#                 yield next(o)
 
def HMM_predict(seq):
    seqlen = len(seq)
    
    #initialize
    trellis = np.zeros((seqlen-1, 3))
    mu = null_dist
    
    #step
    for i in range(1,seqlen):
        aa = seq[i]
        update_step = mu + TM + EM[aa,:]
        mu = update_step.max(axis = 0).reshape(3,1) #need max of every column
        trellis[i-1, : ] = update_step.argmax(axis = 0) #argmax of the column
    
    #last residue
    mu = mu + null_dist
    
    #traceback
    predicted_seq = []
    backpointer = np.argmax(mu)
    
    predicted_seq.append(backpointer)
    
    for i in range(seqlen-2,-1,-1):
        backpointer = int(trellis[i,backpointer])
        predicted_seq.append(backpointer)
    
    predicted_seq = np.array(predicted_seq)
    predicted_seq = np.flip(predicted_seq)
    return(predicted_seq)

# def fasta_predict(input_file)
#     outfile = open('HMM_predict.fasta','w')
#     infile  = open('XXX.fasta','r')
    
#     for line in infile:
#         if line[0] == ">":
#             outfile.write(line)
#             HMM_predict(infile.readline().strip())


In [7]:
HMM_preds = {}
accuracy  = []

testset = np.load('ts1000.npz')

chains = testset.files

for chain in chains:
    HMM_preds[chain] = HMM_predict(testset[chain][0])
    compare = HMM_preds[chain] == testset[chain][1]
    accuracy.append (compare.sum()/len(compare))

np.mean(accuracy)

0.473004160621173

In [82]:
chains

['5d8vA',
 '3nirA',
 '1ucsA',
 '6e6oA',
 '1r6jA',
 '4rekA',
 '4ua6A',
 '3ui4A',
 '2jfrA',
 '7tx0B',
 '4pssA',
 '6etlA',
 '1mc2A',
 '1g6xA',
 '3wdnA',
 '1v6pA',
 '5o99A',
 '4u9hS',
 '7apwA',
 '5x9lA',
 '3vorA',
 '3g21A',
 '1j0pA',
 '5ig6A',
 '6cnwA',
 '1l9lA',
 '2bt9A',
 '2gudA',
 '4a02A',
 '2xjpA',
 '1lugA',
 '3vlaA',
 '3dhaA',
 '3e4gA',
 '4r5rA',
 '1xg0A',
 '3eo6A',
 '3v1aA',
 '4hguA',
 '3ju4A',
 '3q46A',
 '4j5eX',
 '1eb6A',
 '1lniA',
 '5ws7A',
 '1gkmA',
 '2ggcA',
 '1tt8A',
 '2rbkA',
 '5ctmA',
 '3l8wA',
 '2ce2X',
 '4jp6A',
 '3ccdA',
 '5e1yA',
 '1qtwA',
 '1i27A',
 '1m1rA',
 '4xpxA',
 '6dcmA',
 '1i2tA',
 '1d5tA',
 '6s07A',
 '2v3iA',
 '6nnrA',
 '2vzpB',
 '4xotA',
 '6havA',
 '3w42A',
 '4wesB',
 '1w23A',
 '2carA',
 '1c5eA',
 '2aibA',
 '4he6A',
 '2nwfA',
 '1d4tA',
 '1ql0A',
 '1z2uA',
 '3wmuB',
 '2z4uA',
 '1f86A',
 '1uz3A',
 '2fbaA',
 '5d7wA',
 '5jviE',
 '5mx9A',
 '1tukA',
 '4usaA',
 '1u07A',
 '5nb4A',
 '4lwuA',
 '2zk9X',
 '6h8oA',
 '4ku0D',
 '2ciwA',
 '1i4uA',
 '4beuA',
 '6a4uA',
 '6q43A',


In [89]:
accuracy.index(1.0)

99

In [90]:
HMM_preds[chains[99]]

array([2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2], dtype=int64)

In [91]:
testset[chains[99]]

array([[  7,  14,  14,   7,  14,  11,   7,  14,  11,   7,   3,  14,   7,
          3,  14,   7,  14,  14,   7,   0,   1,   7,   5,   0,   7,  19,
         20,   7,  13,  20,   7,  14,  14,   7,  14,  11,   7,  14,  11,
          7,   3,  14,   7,   3,  14,   7],
       [  2,   2,   2,   2,   2,   2,   2,   2,   2,   2,   2,   2,   2,
          2,   2,   2,   2,   2,   2,   2,   2,   2,   2,   2,   2,   2,
          2,   2,   2,   2,   2,   2,   2,   2,   2,   2,   2,   2,   2,
          2,   2,   2,   2,   2,   2,   2],
       [ 96,  99,  89,  10,  77, 108,   8,  75, 110,   6,  83,  82,   9,
         87,  86,   8,  71,  89,  12,  68, 135,  12, 142,  58,   8,  69,
         74,   9, 145,  85,   6,  42,  88,   8,  80, 106,   8,  71, 101,
          7,  92,  77,   6,  92,  96,  48]])